In [ ]:
!pip install sentence-transformers faiss-cpu ollama numpy nltk

In [ ]:
import faiss
import numpy as np
import ollama
import nltk
from sentence_transformers import SentenceTransformer
from typing import List, Dict
from nltk.tokenize import sent_tokenize

In [ ]:
nltk.download("punkt")

In [ ]:
# ------
# CONFIG
# ------
EMBEDDING_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
LLM_MODEL = "llama3.2"

CHUNK_SIZE = 2       # sentences per chunk
CHUNK_OVERLAP = 1    # sentence overlap
TOP_K = 5            # number of chunks to retrieve

In [ ]:
# --------------------------------
# MEDICAL DOCUMENTS (WITH SOURCES)
# --------------------------------
documents = [
    {
        "source": "WHO_Diabetes_Factsheet",
        "text": (
            "Diabetes mellitus is a chronic metabolic disease characterized by elevated levels "
            "of blood glucose. Over time, diabetes can lead to serious damage to the heart, "
            "blood vessels, eyes, kidneys, and nerves. Common symptoms include excessive thirst, "
            "frequent urination, fatigue, and unexplained weight loss."
        )
    },
    {
        "source": "CDC_Hypertension",
        "text": (
            "Hypertension, also known as high blood pressure, is a condition in which the force "
            "of the blood against artery walls is too high. It is often called the silent killer "
            "because many people experience no symptoms. Long-term hypertension increases the "
            "risk of heart disease and stroke."
        )
    },
    {
        "source": "NIH_Asthma",
        "text": (
            "Asthma is a chronic disease of the airways that makes breathing difficult. "
            "Symptoms include wheezing, shortness of breath, chest tightness, and coughing. "
            "Asthma symptoms may vary in frequency and severity."
        )
    },
        {
        "source": "NIH_Anemia",
        "text": (
            "Anemia is a condition in which the body lacks enough healthy red blood cells to "
            "carry adequate oxygen to tissues. Common symptoms include fatigue, weakness, "
            "shortness of breath, and pale skin. Causes of anemia include iron deficiency, "
            "vitamin deficiencies, and chronic diseases."
        )
    },
    {
        "source": "WHO_Heart_Disease",
        "text": (
            "Heart disease refers to a range of conditions affecting the heart, including "
            "coronary artery disease and heart failure. Risk factors include high blood pressure, "
            "high cholesterol, smoking, diabetes, and physical inactivity."
        )
    },
    {
        "source": "CDC_Stroke",
        "text": (
            "A stroke occurs when blood flow to part of the brain is interrupted or reduced, "
            "preventing brain tissue from receiving oxygen and nutrients. Symptoms may include "
            "sudden numbness, confusion, difficulty speaking, and loss of balance."
        )
    },
    {
        "source": "NIH_Migraine",
        "text": (
            "Migraine is a neurological disorder characterized by recurrent headaches that can "
            "be severe and debilitating. Migraines are often accompanied by nausea, vomiting, "
            "and sensitivity to light or sound."
        )
    },
    {
        "source": "WHO_Obesity",
        "text": (
            "Obesity is defined as excessive fat accumulation that presents a risk to health. "
            "It is associated with increased risk of diabetes, heart disease, stroke, and "
            "certain types of cancer."
        )
    },
    {
        "source": "CDC_COVID19",
        "text": (
            "COVID-19 is an infectious disease caused by the SARS-CoV-2 virus. Symptoms range "
            "from mild respiratory illness to severe disease, including fever, cough, "
            "shortness of breath, and loss of taste or smell."
        )
    },
    {
        "source": "NIH_Chronic_Kidney_Disease",
        "text": (
            "Chronic kidney disease is a long-term condition in which the kidneys do not work "
            "effectively. Early stages may have few symptoms, while advanced disease can cause "
            "fatigue, swelling, and changes in urination."
        )
    },
    {
        "source": "WHO_Tuberculosis",
        "text": (
            "Tuberculosis is a bacterial infection that primarily affects the lungs. Symptoms "
            "include persistent cough, chest pain, fever, night sweats, and weight loss."
        )
    },
    {
        "source": "CDC_Depression",
        "text": (
            "Depression is a common mental health disorder characterized by persistent sadness, "
            "loss of interest, and impaired daily functioning. It can also be associated with "
            "changes in appetite, sleep disturbances, and fatigue."
        )
    },
    {
        "source": "NIH_Arthritis",
        "text": (
            "Arthritis refers to inflammation of one or more joints, causing pain and stiffness. "
            "Symptoms often worsen with age, and common forms include osteoarthritis and "
            "rheumatoid arthritis."
        )
    },
    {
        "source": "WHO_Malaria",
        "text": (
            "Malaria is a life-threatening disease caused by parasites transmitted through the "
            "bites of infected mosquitoes. Symptoms typically include fever, chills, headache, "
            "and muscle aches."
        )
    },
    {
        "source": "CDC_Flu",
        "text": (
            "Influenza is a contagious respiratory illness caused by influenza viruses. "
            "Common symptoms include fever, cough, sore throat, muscle aches, and fatigue."
        )
    },
    {
        "source": "NIH_Osteoporosis",
        "text": (
            "Osteoporosis is a bone disease characterized by decreased bone density and "
            "increased fracture risk. It often develops silently over many years without "
            "symptoms until a fracture occurs."
        )
    },
    {
        "source": "WHO_Hepatitis",
        "text": (
            "Hepatitis refers to inflammation of the liver, commonly caused by viral infections. "
            "Symptoms may include fatigue, jaundice, abdominal pain, and nausea."
        )
    },
    {
        "source": "CDC_Smoking",
        "text": (
            "Smoking tobacco is a major risk factor for many diseases, including lung cancer, "
            "heart disease, and chronic respiratory conditions. Quitting smoking significantly "
            "reduces health risks over time."
        )
    },
    {
        "source": "NIH_Alzheimer",
        "text": (
            "Alzheimer’s disease is a progressive neurological disorder that causes memory loss "
            "and cognitive decline. Early symptoms often include difficulty remembering recent "
            "events and conversations."
        )
    },
    {
        "source": "WHO_Nutrition",
        "text": (
            "Good nutrition is essential for maintaining health and preventing disease. "
            "A balanced diet helps reduce the risk of malnutrition, obesity, and chronic "
            "conditions such as heart disease and diabetes."
        )
    },
    {
        "source": "CDC_Allergies",
        "text": (
            "Allergies occur when the immune system reacts to substances that are usually harmless. "
            "Common symptoms include sneezing, itching, runny nose, and watery eyes."
        )
    },
    {
        "source": "NIH_Pneumonia",
        "text": (
            "Pneumonia is an infection that inflames the air sacs in one or both lungs. "
            "Symptoms may include cough, fever, chest pain, and difficulty breathing."
        )
    },
    {
        "source": "WHO_Physical_Activity",
        "text": (
            "Regular physical activity helps improve overall health and reduces the risk of "
            "noncommunicable diseases such as heart disease, diabetes, and certain cancers."
        )
    }
]

In [ ]:
# --------
# CHUNKING
# --------
def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    sentences = sent_tokenize(text)
    chunks = []

    i = 0
    while i < len(sentences):
        chunk = sentences[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        i += chunk_size - overlap

    return chunks

chunks = []
metadata = []

for doc in documents:
    doc_chunks = chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)
    for idx, chunk in enumerate(doc_chunks):
        chunks.append(chunk)
        metadata.append({
            "source": doc["source"],
            "chunk_id": idx
        })

print(f"Total chunks created: {len(chunks)}")

In [ ]:
# ----------
# EMBEDDINGS
# ----------
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
)

dim = embeddings.shape[1]

In [ ]:
embeddings

In [ ]:
# -----------
# FAISS INDEX
# -----------
index = faiss.IndexFlatIP(dim)  # cosine similarity
index.add(embeddings)

print("FAISS index size:", index.ntotal)

In [ ]:
# ---------
# RETRIEVER
# ---------
def retrieve(query: str, top_k: int) -> List[Dict]:
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx],
            "score": float(score),
            "source": metadata[idx]["source"],
            "chunk_id": metadata[idx]["chunk_id"]
        })

    return results

In [ ]:
# ---------------------
# PROMPT WITH CITATIONS
# ---------------------
def build_prompt(query: str, retrieved_docs: List[Dict]) -> str:
    context_blocks = []

    for i, doc in enumerate(retrieved_docs):
        citation = f"[{i+1}] Source: {doc['source']} | Chunk: {doc['chunk_id']}"
        context_blocks.append(f"{citation}\n{doc['text']}")

    context = "\n\n".join(context_blocks)

    prompt = f"""
        You are a medical information assistant.
        
        Answer the question ONLY using the context provided.
        Each factual statement MUST include a citation like [1], [2].
        If the answer is not contained in the context, say:
        "I don't know based on the provided information."
        
        Do NOT provide diagnosis or treatment advice.
        
        Context:
        {context}
        
        Question:
        {query}
        
        Answer:
    """
    return prompt.strip()

In [ ]:
# -------------------
# GENERATION (OLLAMA)
# -------------------
def generate(prompt: str) -> str:
    response = ollama.generate(
        model = LLM_MODEL,
        prompt = prompt,
        options = {
            "temperature": 0.2
        }
    )
    return response["response"]

In [ ]:
# -----------------
# FULL RAG PIPELINE
# -----------------
def medical_rag(query: str):
    retrieved = retrieve(query, TOP_K)
    prompt = build_prompt(query, retrieved)
    answer = generate(prompt)

    return {
        "query": query,
        "retrieved_chunks": retrieved,
        "answer": answer
    }

In [ ]:
# ----------
# USER QUERY
# ----------
while True:
    query = input("Question: ") # "What are common symptoms of diabetes?"
    if query.lower() in ["quit","exit"]: break
    result = medical_rag(query)

    print("\n--- RETRIEVED CHUNKS ---")
    for r in result["retrieved_chunks"]:
        print(f"- ({r['score']:.3f}) {r['source']} | Chunk {r['chunk_id']}")
        print(f"  Content: {r['text']}\n")  # <-- added to show actual text

    print("\n--- ANSWER ---")
    print(result["answer"])

    print("\n*****Disclaimer: This information is for educational purposes only and does not replace professional medical advice.*****")

## Sample Questions:
- frequent urination
- fever
- fever and cough

# Using Cosine-Similarity and Maximal Marginal Relevance (MMR) for Retriever

In [ ]:
import faiss
import numpy as np
import ollama
import nltk
from sentence_transformers import SentenceTransformer
from typing import List, Dict

nltk.download("punkt")
from nltk.tokenize import sent_tokenize

# ------
# CONFIG
# ------
EMBEDDING_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
LLM_MODEL = "llama3.2"

CHUNK_SIZE = 2       # sentences per chunk
CHUNK_OVERLAP = 1    # sentence overlap
TOP_K = 5
MMR_LAMBDA = 0.5     # balance relevance vs diversity

# --------------------------------------------
# MEDICAL DOCUMENTS (same structure as before)
# --------------------------------------------
documents = [
    {
        "source": "WHO_Diabetes_Factsheet",
        "text": (
            "Diabetes mellitus is a chronic metabolic disease characterized by elevated levels "
            "of blood glucose. Over time, diabetes can lead to serious damage to the heart, "
            "blood vessels, eyes, kidneys, and nerves. Common symptoms include excessive thirst, "
            "frequent urination, fatigue, and unexplained weight loss."
        )
    },
    {
        "source": "CDC_Hypertension",
        "text": (
            "Hypertension, also known as high blood pressure, is a condition in which the force "
            "of the blood against artery walls is too high. It is often called the silent killer "
            "because many people experience no symptoms. Long-term hypertension increases the "
            "risk of heart disease and stroke."
        )
    },
    {
        "source": "NIH_Asthma",
        "text": (
            "Asthma is a chronic disease of the airways that makes breathing difficult. "
            "Symptoms include wheezing, shortness of breath, chest tightness, and coughing. "
            "Asthma symptoms may vary in frequency and severity."
        )
    },
        {
        "source": "NIH_Anemia",
        "text": (
            "Anemia is a condition in which the body lacks enough healthy red blood cells to "
            "carry adequate oxygen to tissues. Common symptoms include fatigue, weakness, "
            "shortness of breath, and pale skin. Causes of anemia include iron deficiency, "
            "vitamin deficiencies, and chronic diseases."
        )
    },
    {
        "source": "WHO_Heart_Disease",
        "text": (
            "Heart disease refers to a range of conditions affecting the heart, including "
            "coronary artery disease and heart failure. Risk factors include high blood pressure, "
            "high cholesterol, smoking, diabetes, and physical inactivity."
        )
    },
    {
        "source": "CDC_Stroke",
        "text": (
            "A stroke occurs when blood flow to part of the brain is interrupted or reduced, "
            "preventing brain tissue from receiving oxygen and nutrients. Symptoms may include "
            "sudden numbness, confusion, difficulty speaking, and loss of balance."
        )
    },
    {
        "source": "NIH_Migraine",
        "text": (
            "Migraine is a neurological disorder characterized by recurrent headaches that can "
            "be severe and debilitating. Migraines are often accompanied by nausea, vomiting, "
            "and sensitivity to light or sound."
        )
    },
    {
        "source": "WHO_Obesity",
        "text": (
            "Obesity is defined as excessive fat accumulation that presents a risk to health. "
            "It is associated with increased risk of diabetes, heart disease, stroke, and "
            "certain types of cancer."
        )
    },
    {
        "source": "CDC_COVID19",
        "text": (
            "COVID-19 is an infectious disease caused by the SARS-CoV-2 virus. Symptoms range "
            "from mild respiratory illness to severe disease, including fever, cough, "
            "shortness of breath, and loss of taste or smell."
        )
    },
    {
        "source": "NIH_Chronic_Kidney_Disease",
        "text": (
            "Chronic kidney disease is a long-term condition in which the kidneys do not work "
            "effectively. Early stages may have few symptoms, while advanced disease can cause "
            "fatigue, swelling, and changes in urination."
        )
    },
    {
        "source": "WHO_Tuberculosis",
        "text": (
            "Tuberculosis is a bacterial infection that primarily affects the lungs. Symptoms "
            "include persistent cough, chest pain, fever, night sweats, and weight loss."
        )
    },
    {
        "source": "CDC_Depression",
        "text": (
            "Depression is a common mental health disorder characterized by persistent sadness, "
            "loss of interest, and impaired daily functioning. It can also be associated with "
            "changes in appetite, sleep disturbances, and fatigue."
        )
    },
    {
        "source": "NIH_Arthritis",
        "text": (
            "Arthritis refers to inflammation of one or more joints, causing pain and stiffness. "
            "Symptoms often worsen with age, and common forms include osteoarthritis and "
            "rheumatoid arthritis."
        )
    },
    {
        "source": "WHO_Malaria",
        "text": (
            "Malaria is a life-threatening disease caused by parasites transmitted through the "
            "bites of infected mosquitoes. Symptoms typically include fever, chills, headache, "
            "and muscle aches."
        )
    },
    {
        "source": "CDC_Flu",
        "text": (
            "Influenza is a contagious respiratory illness caused by influenza viruses. "
            "Common symptoms include fever, cough, sore throat, muscle aches, and fatigue."
        )
    },
    {
        "source": "NIH_Osteoporosis",
        "text": (
            "Osteoporosis is a bone disease characterized by decreased bone density and "
            "increased fracture risk. It often develops silently over many years without "
            "symptoms until a fracture occurs."
        )
    },
    {
        "source": "WHO_Hepatitis",
        "text": (
            "Hepatitis refers to inflammation of the liver, commonly caused by viral infections. "
            "Symptoms may include fatigue, jaundice, abdominal pain, and nausea."
        )
    },
    {
        "source": "CDC_Smoking",
        "text": (
            "Smoking tobacco is a major risk factor for many diseases, including lung cancer, "
            "heart disease, and chronic respiratory conditions. Quitting smoking significantly "
            "reduces health risks over time."
        )
    },
    {
        "source": "NIH_Alzheimer",
        "text": (
            "Alzheimer’s disease is a progressive neurological disorder that causes memory loss "
            "and cognitive decline. Early symptoms often include difficulty remembering recent "
            "events and conversations."
        )
    },
    {
        "source": "WHO_Nutrition",
        "text": (
            "Good nutrition is essential for maintaining health and preventing disease. "
            "A balanced diet helps reduce the risk of malnutrition, obesity, and chronic "
            "conditions such as heart disease and diabetes."
        )
    },
    {
        "source": "CDC_Allergies",
        "text": (
            "Allergies occur when the immune system reacts to substances that are usually harmless. "
            "Common symptoms include sneezing, itching, runny nose, and watery eyes."
        )
    },
    {
        "source": "NIH_Pneumonia",
        "text": (
            "Pneumonia is an infection that inflames the air sacs in one or both lungs. "
            "Symptoms may include cough, fever, chest pain, and difficulty breathing."
        )
    },
    {
        "source": "WHO_Physical_Activity",
        "text": (
            "Regular physical activity helps improve overall health and reduces the risk of "
            "noncommunicable diseases such as heart disease, diabetes, and certain cancers."
        )
    }
]

# --------
# CHUNKING
# --------
def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    sentences = sent_tokenize(text)
    chunks = []

    i = 0
    while i < len(sentences):
        chunk = sentences[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        i += chunk_size - overlap

    return chunks

chunks = []
metadata = []

for doc in documents:
    doc_chunks = chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)
    for idx, chunk in enumerate(doc_chunks):
        chunks.append(chunk)
        metadata.append({
            "source": doc["source"],
            "chunk_id": idx
        })

print(f"Total chunks created: {len(chunks)}")

# ----------
# EMBEDDINGS
# ----------
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
)

dim = embeddings.shape[1]

# -----------
# FAISS INDEX
# -----------
index = faiss.IndexFlatIP(dim)  # inner product = cosine similarity
index.add(embeddings)
print("FAISS index size:", index.ntotal)

# -------------
# MMR RETRIEVAL
# The following function calculates MMR over ALL chunks, which may be slow if your number of chunks is large
# -------------
# def mmr(query: str, top_k: int = TOP_K, lambda_param: float = MMR_LAMBDA) -> List[Dict]:
#     query_emb = embedding_model.encode([query], normalize_embeddings=True)

#     # Compute cosine similarity to all chunks
#     scores = (embeddings @ query_emb.T).flatten()  # cosine similarity since embeddings normalized

#     selected = []
#     selected_indices = []
#     candidate_indices = list(range(len(chunks)))

#     while len(selected) < top_k and candidate_indices:
#         mmr_score = []
#         for i in candidate_indices:
#             if not selected:
#                 # no selected chunks yet → just relevance
#                 score = lambda_param * scores[i]
#             else:
#                 # penalize similarity with already selected chunks
#                 max_sim = max([np.dot(embeddings[i], embeddings[j]) for j in selected_indices])
#                 score = lambda_param * scores[i] - (1 - lambda_param) * max_sim
#             mmr_score.append(score)

#         # pick chunk with max MMR score
#         max_idx = candidate_indices[np.argmax(mmr_score)]
#         selected.append({
#             "text": chunks[max_idx],
#             "score": float(scores[max_idx]),
#             "source": metadata[max_idx]["source"],
#             "chunk_id": metadata[max_idx]["chunk_id"]
#         })
#         selected_indices.append(max_idx)
#         candidate_indices.remove(max_idx)

#     return selected

# -------------
# MMR RETRIEVAL
# The following function first calculates the cosine similarity for all chunks, then calculates MMR over the top n chunks
# -------------
def mmr_with_candidates(query: str, candidate_top_n: int = 10, select_k: int = 3, lambda_param: float = MMR_LAMBDA):
    query_emb = embedding_model.encode([query], normalize_embeddings=True)
    
    # Step 1: cosine similarity to all chunks
    scores = (embeddings @ query_emb.T).flatten()
    
    # Get top N candidates by cosine similarity
    candidate_indices = np.argsort(-scores)[:candidate_top_n].tolist()
    
    selected = []
    selected_indices = []

    while len(selected) < select_k and candidate_indices:
        mmr_score = []
        for i in candidate_indices:
            if not selected:
                score = lambda_param * scores[i]
            else:
                max_sim = max([np.dot(embeddings[i], embeddings[j]) for j in selected_indices])
                score = lambda_param * scores[i] - (1 - lambda_param) * max_sim
            mmr_score.append(score)
        
        max_idx = candidate_indices[np.argmax(mmr_score)]
        selected.append({
            "text": chunks[max_idx],
            "score": float(scores[max_idx]),
            "source": metadata[max_idx]["source"],
            "chunk_id": metadata[max_idx]["chunk_id"]
        })
        selected_indices.append(max_idx)
        candidate_indices.remove(max_idx)

    return selected

# ---------------------
# PROMPT WITH CITATIONS
# ---------------------
def build_prompt(query: str, retrieved_docs: List[Dict]) -> str:
    context_blocks = []
    for i, doc in enumerate(retrieved_docs):
        citation = f"[{i+1}] Source: {doc['source']} | Chunk: {doc['chunk_id']}"
        context_blocks.append(f"{citation}\n{doc['text']}")

    context = "\n\n".join(context_blocks)
    prompt = f"""
        You are a medical information assistant.
        
        Answer the question ONLY using the context provided.
        Each factual statement MUST include a citation like [1], [2].
        If the answer is not contained in the context, say:
        "I don't know based on the provided information."
        
        Do NOT provide diagnosis or treatment advice.
        
        Context:
        {context}
        
        Question:
        {query}
        
        Answer:
    """
    return prompt.strip()

# -------------------
# GENERATION (OLLAMA)
# -------------------
def generate(prompt: str) -> str:
    response = ollama.generate(
        model=LLM_MODEL,
        prompt=prompt,
        options={"temperature": 0.2}
    )
    return response["response"]

# -----------------
# FULL RAG PIPELINE
# -----------------
def medical_rag(query: str):
    # retrieved = mmr(query, top_k=TOP_K, lambda_param=MMR_LAMBDA)
    retrieved = mmr_with_candidates(query, 
                                    candidate_top_n = 10, 
                                    select_k = TOP_K, 
                                    lambda_param = MMR_LAMBDA)
    prompt = build_prompt(query, retrieved)
    answer = generate(prompt)
    return {
        "query": query,
        "retrieved_chunks": retrieved,
        "answer": answer
    }

# ----------------------
# INTERACTIVE QUERY LOOP
# ----------------------
while True:
    query = input("Question: ")
    if query.lower() in ["quit", "exit"]:
        break

    result = medical_rag(query)

    print("\n--- RETRIEVED CHUNKS ---")
    for r in result["retrieved_chunks"]:
        print(f"- ({r['score']:.3f}) {r['source']} | Chunk {r['chunk_id']}")
        print(f"  Content: {r['text']}\n")

    print("\n--- ANSWER ---")
    print(result["answer"])
    
    print("\n*****Disclaimer: This information is for educational purposes only and does not replace professional medical advice.*****")

# Visualizing Cosine-Similarity and MMR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# -----------------------------------------
# Reduce embeddings to 2D for visualization
# -----------------------------------------
tsne = TSNE(n_components=2, random_state=42)
emb_2d = tsne.fit_transform(embeddings)

# -------------
# Example query
# -------------
query = "What are the complications of diabetes?"
query_emb = embedding_model.encode([query], normalize_embeddings=True)

# Compute cosine similarity
scores = (embeddings @ query_emb.T).flatten()

# ----------------------------
# Cosine similarity selections
# ----------------------------
# Top 10 candidates
top10_cosine_idx = np.argsort(-scores)[:10]

# Top 3 for cosine
top3_cosine_idx = top10_cosine_idx[:3]

# --------------------------------------
# MMR on top 10 candidates, select top 3
# --------------------------------------
candidate_indices = list(top10_cosine_idx)
selected = []
selected_indices = []

lambda_param = 0.5
select_k = 3

while len(selected) < select_k and candidate_indices:
    mmr_score = []
    for i in candidate_indices:
        if not selected:
            score = lambda_param * scores[i]
        else:
            max_sim = max([float(np.dot(embeddings[i], embeddings[j])) for j in selected_indices])
            score = lambda_param * scores[i] - (1 - lambda_param) * max_sim
        mmr_score.append(score)
    
    max_idx = candidate_indices[np.argmax(mmr_score)]
    selected.append(max_idx)
    selected_indices.append(max_idx)
    candidate_indices.remove(max_idx)

# --------
# Plotting
# --------
plt.figure(figsize=(10,7))

# All chunks
plt.scatter(emb_2d[:,0], emb_2d[:,1], c='lightgray', label='All chunks')

# Top 10 cosine candidates
plt.scatter(emb_2d[top10_cosine_idx,0], emb_2d[top10_cosine_idx,1], 
            c='orange', label='Top 10 cosine', s=80)

# Top 3 cosine similarity
plt.scatter(emb_2d[top3_cosine_idx,0], emb_2d[top3_cosine_idx,1], 
            c='red', label='Top 3 cosine', s=120, edgecolors='black')

# Top 3 MMR
plt.scatter(emb_2d[selected,0], emb_2d[selected,1], 
            c='blue', label='Top 3 MMR', s=120, marker='X')

# Query (we need to transform only query embedding to 2D relative to corpus)
tsne_query = TSNE(n_components=2, random_state=42)
query_2d = tsne_query.fit_transform(np.vstack([embeddings, query_emb]))[-1]
plt.scatter(query_2d[0], query_2d[1], c='green', label='Query', s=200, marker='*')

plt.title("Cosine Similarity vs MMR Selection with Top 10 Candidates")
plt.legend()
plt.show()

## Explanation
- If you simply perform cosine-similarity and take the top-3, the 3 points will contain a lot of overlaps.
- A better option is to perform cosine-similarity and take the top-10, and then perform MMR and take the top 3. This way, the results would not contain so much overlap and the there would be more diversity in the result.